In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
file_path = '/content/drive/MyDrive/final_mta_ml_matrix.csv' # adjust as needed

In [ ]:
mta_df = pd.read_csv(file_path)

In [ ]:
mta_df = mta_df.drop(columns=['transit_timestamp'])

In [ ]:
X = mta_df.drop(columns=['ridership'])
y = mta_df['ridership'].values.reshape(-1, 1)

In [ ]:
split_idx = int(len(mta_df) * 0.8)

X_train_pd, X_test_pd = X.iloc[:split_idx], X.iloc[split_idx:]
y_train_np, y_test_np = y[:split_idx], y[split_idx:]

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

In [ ]:
cont_cols = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'lag_1h', 'lag_24h']

# Fit/transform on train, only transform on test
X_train_pd[cont_cols] = scaler_X.fit_transform(X_train_pd[cont_cols])
X_test_pd[cont_cols] = scaler_X.transform(X_test_pd[cont_cols])

# Scale target
y_train_np = scaler_y.fit_transform(y_train_np)
y_test_np = scaler_y.transform(y_test_np)

In [ ]:
print(X_train_pd.columns)

In [ ]:
X_train = torch.tensor(X_train_pd.values, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32)

X_test = torch.tensor(X_test_pd.values, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
class MTARidershipNN(nn.Module):
    def __init__(self, input_dim):
        super(MTARidershipNN, self).__init__()

        # The "Funnel" Architecture
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4), # Regularization to prevent overfitting to noisy surges

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Linear(16, 1) # Linear output for continuous ridership volume
        )

    def forward(self, x):
        return self.network(x)

# Initialize the model
model = MTARidershipNN(input_dim=38)

In [ ]:
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
epochs = 100
patience = 15

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_val_batch, y_val_batch in test_loader:
            val_predictions = model(X_val_batch)
            loss = criterion(val_predictions, y_val_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)

    scheduler.step(avg_val_loss)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}]  Train Loss: {avg_loss:.4f}  Val Loss: {avg_val_loss:.4f}')

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}')
            break

# Restore best weights before evaluation
model.load_state_dict(best_model_state)

In [ ]:
model.eval()
all_preds, all_actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch)
        all_preds.append(preds.numpy())
        all_actuals.append(y_batch.numpy())

preds_orig = scaler_y.inverse_transform(np.concatenate(all_preds))
actuals_orig = scaler_y.inverse_transform(np.concatenate(all_actuals))

mae = np.mean(np.abs(preds_orig - actuals_orig))
rmse = np.sqrt(np.mean((preds_orig - actuals_orig)**2))
wmape = np.sum(np.abs(preds_orig - actuals_orig)) / np.sum(np.abs(actuals_orig))

In [ ]:
print(f"mae: {mae}")
print(f"rmse: {rmse}")
print(f"wmape: {wmape}")

In [ ]:
results_df = pd.DataFrame({'actual': actuals_orig.flatten(), 'pred': preds_orig.flatten()})
results_df['hour'] = X_test_pd['hour'].values  # or however hour is stored
results_df['abs_error'] = np.abs(results_df['actual'] - results_df['pred'])

print(results_df.groupby('hour')['abs_error'].mean())

In [ ]:
import matplotlib.pyplot as plt
results_df.groupby('hour')['abs_error'].mean().plot(kind='bar')
plt.title('Mean Absolute Error by Hour of Day')
plt.xlabel('Hour'); plt.ylabel('MAE (riders)')
plt.tight_layout()

In [ ]:
mta_df.groupby('hour')['ridership'].mean()

## Log-Transform Target Experiment

In [ ]:
X = mta_df.drop(columns=['ridership'])
y = np.log1p(mta_df['ridership'].values.reshape(-1, 1))

split_idx = int(len(mta_df) * 0.8)

X_train_pd, X_test_pd = X.iloc[:split_idx], X.iloc[split_idx:]
y_train_np, y_test_np = y[:split_idx], y[split_idx:]

scaler_X = StandardScaler()

cont_cols = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'lag_1h', 'lag_24h']

# Fit/transform on train, only transform on test
X_train_pd[cont_cols] = scaler_X.fit_transform(X_train_pd[cont_cols])
X_test_pd[cont_cols] = scaler_X.transform(X_test_pd[cont_cols])

# Scale target
y_train_np = scaler_y.fit_transform(y_train_np)
y_test_np = scaler_y.transform(y_test_np)

In [ ]:
# Initialize the model
model = MTARidershipNN(input_dim=38)

criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
epochs = 100
patience = 15

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_val_batch, y_val_batch in test_loader:
            val_predictions = model(X_val_batch)
            loss = criterion(val_predictions, y_val_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)

    scheduler.step(avg_val_loss)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}]  Train Loss: {avg_loss:.4f}  Val Loss: {avg_val_loss:.4f}')

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}')
            break

# Restore best weights before evaluation
model.load_state_dict(best_model_state)

In [ ]:
model.eval()
all_preds, all_actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch)
        all_preds.append(preds.numpy())
        all_actuals.append(y_batch.numpy())

# updated inverse transform
preds_orig = np.expm1(scaler_y.inverse_transform(np.concatenate(all_preds)))
actuals_orig = np.expm1(scaler_y.inverse_transform(np.concatenate(all_actuals)))

mae = np.mean(np.abs(preds_orig - actuals_orig))
rmse = np.sqrt(np.mean((preds_orig - actuals_orig)**2))
wmape = np.sum(np.abs(preds_orig - actuals_orig)) / np.sum(np.abs(actuals_orig))

In [ ]:
print(f"mae: {mae}")
print(f"rmse: {rmse}")
print(f"wmape: {wmape}")

## individual station comparison

In [ ]:
hub_cols = ['hub_name_Fulton Center', 'hub_name_Grand Central',
            'hub_name_Herald Square', 'hub_name_Penn Station',
            'hub_name_Times Sq', 'hub_name_WTC']

results_df['station'] = X_test_pd[hub_cols].values.argmax(axis=1)
results_df['station'] = results_df['station'].map({i: col for i, col in enumerate(hub_cols)})

station_metrics = results_df.groupby('station').apply(lambda df: pd.Series({
    'MAE': np.mean(np.abs(df['actual'] - df['pred'])),
    'WMAPE': np.sum(np.abs(df['actual'] - df['pred'])) / np.sum(np.abs(df['actual'])),
    'avg_actual': df['actual'].mean()
}), include_groups=False).reset_index()

print(station_metrics.sort_values('WMAPE'))

In [ ]:
print(mta_df.iloc[split_idx:][['hub_name_Times Sq', 'hub_name_WTC',
                                'hub_name_Fulton Center', 'hub_name_Grand Central',
                                'hub_name_Herald Square', 'hub_name_Penn Station']].sum())

In [ ]:
print(mta_df[['hub_name_Times Sq', 'hub_name_WTC',
                                'hub_name_Fulton Center', 'hub_name_Grand Central',
                                'hub_name_Herald Square', 'hub_name_Penn Station']].sum())

# Fixed time series splitting

In [ ]:
mta_df = pd.read_csv(file_path)
mta_df = mta_df.drop(columns=['transit_timestamp'])

X = mta_df.drop(columns=['ridership'])
y = mta_df['ridership'].values.reshape(-1, 1)

hub_cols = ['hub_name_Fulton Center', 'hub_name_Grand Central',
            'hub_name_Herald Square', 'hub_name_Penn Station',
            'hub_name_Times Sq', 'hub_name_WTC']

train_dfs, test_dfs = [], []
for _, station_df in mta_df.groupby(hub_cols):
    split = int(len(station_df) * 0.8)
    train_dfs.append(station_df.iloc[:split])
    test_dfs.append(station_df.iloc[split:])

train_df = pd.concat(train_dfs).reset_index(drop=True)
test_df = pd.concat(test_dfs).reset_index(drop=True)

X_train_pd = train_df.drop(columns=['ridership'])
X_test_pd = test_df.drop(columns=['ridership'])
y_train_np = train_df['ridership'].values.reshape(-1, 1)
y_test_np = test_df['ridership'].values.reshape(-1, 1)


scaler_X = StandardScaler()
scaler_y = StandardScaler()

cont_cols = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'lag_1h', 'lag_24h']

# Fit/transform on train, only transform on test
X_train_pd[cont_cols] = scaler_X.fit_transform(X_train_pd[cont_cols])
X_test_pd[cont_cols] = scaler_X.transform(X_test_pd[cont_cols])

# Scale target
y_train_np = scaler_y.fit_transform(y_train_np)
y_test_np = scaler_y.transform(y_test_np)


X_train = torch.tensor(X_train_pd.values, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32)

X_test = torch.tensor(X_test_pd.values, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


# Initialize the model
model = MTARidershipNN(input_dim=38)

criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
epochs = 100
patience = 15

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_val_batch, y_val_batch in test_loader:
            val_predictions = model(X_val_batch)
            loss = criterion(val_predictions, y_val_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)

    scheduler.step(avg_val_loss)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}]  Train Loss: {avg_loss:.4f}  Val Loss: {avg_val_loss:.4f}')

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}')
            break

# Restore best weights before evaluation
model.load_state_dict(best_model_state)

In [ ]:
model.eval()
all_preds, all_actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch)
        all_preds.append(preds.numpy())
        all_actuals.append(y_batch.numpy())

preds_orig = scaler_y.inverse_transform(np.concatenate(all_preds))
actuals_orig = scaler_y.inverse_transform(np.concatenate(all_actuals))

mae = np.mean(np.abs(preds_orig - actuals_orig))
rmse = np.sqrt(np.mean((preds_orig - actuals_orig)**2))
wmape = np.sum(np.abs(preds_orig - actuals_orig)) / np.sum(np.abs(actuals_orig))

In [ ]:
print(f"mae: {mae}")
print(f"rmse: {rmse}")
print(f"wmape: {wmape}")

In [ ]:
results_df = pd.DataFrame({'actual': actuals_orig.flatten(), 'pred': preds_orig.flatten()})
results_df['station'] = X_test_pd[hub_cols].values.argmax(axis=1)
results_df['station'] = results_df['station'].map({i: col for i, col in enumerate(hub_cols)})
results_df['abs_error'] = np.abs(results_df['actual'] - results_df['pred'])

station_metrics = results_df.groupby('station').apply(lambda df: pd.Series({
    'MAE': np.mean(np.abs(df['actual'] - df['pred'])),
    'WMAPE': np.sum(np.abs(df['actual'] - df['pred'])) / np.sum(np.abs(df['actual'])),
    'avg_actual': df['actual'].mean()
}), include_groups=False).reset_index()

print(station_metrics.sort_values('WMAPE'))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

hourly_mae = results_df.groupby('hour')['abs_error'].mean()
hourly_avg_ridership = mta_df.groupby('hour')['ridership'].mean()

fig, ax1 = plt.subplots()

# Bar chart on left axis (MAE)
ax1.bar(hourly_mae.index, hourly_mae.values, color='blue', edgecolor='white', label='MAE')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Mean Absolute Error (riders)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticks(range(24))

# Line chart on right axis (average ridership)
ax2 = ax1.twinx()
ax2.plot(hourly_avg_ridership.index, hourly_avg_ridership.values,
         color='red', linewidth=2, marker='o', markersize=4, label='Avg Ridership')
ax2.set_ylabel('Average Actual Ridership (riders)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('Neural Network: Mean Absolute Error by Hour of Day (Fixed Split)')
plt.tight_layout()
plt.savefig('nn_residual_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results_df['signed_error'] = results_df['pred'] - results_df['actual']
results_df.groupby('hour')['signed_error'].mean()

## LSTM test

In [ ]:
class MTARidershipLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super(MTARidershipLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = x.unsqueeze(1)  # reshape to (batch, seq_len=1, features)
        lstm_out, _ = self.lstm(x)
        return self.output(lstm_out[:, -1, :])  # take last timestep

# Initialize
lstm_model = MTARidershipLSTM(input_dim=38)

criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Training loop (identical to MLP)
epochs = 100
patience = 15
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    lstm_model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = lstm_model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    lstm_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_val_batch, y_val_batch in test_loader:
            val_predictions = lstm_model(X_val_batch)
            loss = criterion(val_predictions, y_val_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)
    scheduler.step(avg_val_loss)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}]  Train Loss: {avg_loss:.4f}  Val Loss: {avg_val_loss:.4f}')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_state = lstm_model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}')
            break

lstm_model.load_state_dict(best_model_state)

# Evaluation
lstm_model.eval()
all_preds, all_actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = lstm_model(X_batch)
        all_preds.append(preds.numpy())
        all_actuals.append(y_batch.numpy())

preds_orig = scaler_y.inverse_transform(np.concatenate(all_preds))
actuals_orig = scaler_y.inverse_transform(np.concatenate(all_actuals))

mae = np.mean(np.abs(preds_orig - actuals_orig))
rmse = np.sqrt(np.mean((preds_orig - actuals_orig)**2))
wmape = np.sum(np.abs(preds_orig - actuals_orig)) / np.sum(np.abs(actuals_orig))

print(f"LSTM MAE: {mae:.1f}")
print(f"LSTM RMSE: {rmse:.1f}")
print(f"LSTM WMAPE: {wmape:.4f}")